# S6E8 — Lookup-Transformer + Insights（解説付き学習ノート）| 項目 | 内容 ||---|---|| コンペ | [Predicting Smartphone Addiction (Playground Series S6E8)](https://www.kaggle.com/competitions/playground-series-s6e8) || 原著notebook | [S6E8 \| Lookup-Transformer + Insights lb 0.97041](https://www.kaggle.com/code/tamerlanomralinov/s6e8-lookup-transformer-insights-lb-0-97041) || 原著者 | Tamerlan Omralinov || スコア | Public LB 0.97041（30 votes） || ライセンス | Apache 2.0 |> **断り書き**: これは**学習目的の解説付き写し**です。原著のコードは一切変更しておらず、出力（outputs）だけを削除して、> 各コードセルの直前に日本語の解説Markdownセルを挿入しています。功績はすべて原著者に帰属します。## なぜこの notebook を選んだか本日の公開LB最上位は 0.97099 の notebook でしたが、中身は**他人の提出CSV3本をrank平均しただけの4セル**で、学べることがほとんどありませんでした。一方こちらは 0.97041（差は 0.0006）ながら、**データ生成過程そのものを実験で突き止めて、そこに合わせたモデルをゼロから設計している**という、Kaggleで最も価値のある種類の notebook です。スコアの0.0006より、この思考プロセスのほうが遥かに学ぶ価値があります。## 手法の概要 — 2つの「発見」がすべてこのコンペのデータは、小さな実データから生成された**合成データ（synthetic data）**です。原著者はそこに2つの構造を見つけました。**発見1: 値は「大きさ」ではなく「照合キー」である。**`notifications_per_day` は単変量AUCが **0.492**（＝単調な信号がまったくない）のに、データを独立な2つの半分に割って**値ごとの残差**を計算すると、その相関が **0.72** もある。つまり「通知数が多いほど依存的」のような単調関係はないのに、「通知数がちょうど 47.00 の人はラベルが高い」という**値そのものへの紐付け**が存在する。合成データの生成器が、元データの値→ラベル対応を丸暗記してしまったためです。各列は2桁に量子化され、異なる値は167〜1460種類しかなく、train/testでほぼ共通。→ **各列の「厳密な値」に対する埋め込みテーブル（学習可能なルックアップ表）を持たせるべき**。**発見2: 生成過程に厳密な制約がある。**`daily_screen_time_hours ≥ social + gaming + work` が **859,029行の100.00000%** で成立し、最小ギャップはぴったり0.000。つまりこの3列は総スクリーンタイムの**構成要素**であり、その差分（＝説明のつかないスクリーンタイム）は実在する潜在変数（単独AUC 0.765）。GBDTは4項の引き算を木の分割で作れないので、**明示的に特徴量として与える**必要がある。## モデル: Lookup-Transformer各列が1つの「トークン」になります。```token_j = Embedding_j[厳密な値のID]  +  PLR_j(rank-gauss変換した値) × (1 - 欠損フラグ)```- 前半 `Embedding_j[...]` が**発見1に対応する学習可能なルックアップ表**- 後半 `PLR_j(...)` は **Periodic-Linear 埋め込み**。学習可能なフーリエ周波数で数値を sin/cos に展開してから線形変換するもので、  生のスカラーをMLPに突っ込むより滑らかな傾向を遥かに効率よく表現できる- 欠損時は後半を切り、`Embedding_j[0]` だけが残る → **「欠損」自体が列ごとに固有の表現を持つ**（補完しない）- 最後に CLS トークンが全体を読み出す（Transformerの標準的な集約方法）さらに CatBoost と LightGBM を「相棒」として学習し、OOFで重み探索した**rankブレンド**で提出します。## 評価指標（Evaluation Metric）**タスク**: スマートフォン依存（`addicted_label`）の二値分類。**指標**: **ROC-AUC**（Area Under the ROC Curve）。- **意味**: 「ランダムに選んだ陽性1件と陰性1件について、モデルが陽性により高いスコアを与える確率」。  0.5 が当てずっぽう、1.0 が完璧。- **計算**: 閾値を0→1まで動かしながら (偽陽性率, 真陽性率) をプロットした曲線の下の面積。- **なぜこの指標か**: AUCは**予測値の順位だけ**を見ます。絶対的な確率のキャリブレーションを問わないので、  クラス不均衡があっても安定して評価でき、閾値を決め打ちする必要もありません。  「依存傾向の高い順にユーザーを並べたい」という実務的な使い道とも一致します。**この notebook の設計が AUC をどう最適化しているか**:- **rankブレンドを使う**: AUCが順位しか見ないので、複数モデルを混ぜるときは  生の確率を平均するより `rankdata(v)/len(v)` で**順位に正規化してから平均**するのが理にかなっています。  スケールの違うモデル同士でも公平に混ざる。最後に min-max で [0,1] に戻していますが、  これは単調変換なのでAUCは変わりません（提出形式を整えているだけ）。- **損失関数は BCEWithLogitsLoss**: AUCは微分不可能なので直接最適化できません。  代わりに対数損失を最適化し、**検証はAUCで行う**（`roc_auc_score` でベストエポックを選ぶ）という  「代理損失で学習・本番指標で選択」の定石を踏んでいます。- **重み探索をOOF上で行う**: 全モデルが同じfold分割を共有しているため、  ブレンド重みの推定が正直（leakageがない）。ここを崩すとLBで必ず裏切られます。- **11-fold の StratifiedKFold**: fold数を増やすと各foldの検証セットが小さくなる代わりに学習データが増え、  OOF全体としてのAUC推定が安定します。

# S6E8 — Lookup-Transformer

A custom deep-learning solution for *Predicting Smartphone Addiction*, designed around two
properties of this dataset that ordinary tabular models do not exploit.

**Finding 1 — the exact value is a lookup key, not a magnitude.**
The synthetic generator memorised value→label associations from the small original dataset.
`notifications_per_day` has univariate AUC **0.492** (no monotone signal at all), yet its
per-value residuals correlate **0.72** across two independent halves of the data. The columns
are quantised to 2 decimals with only 167–1460 distinct values, and train/test share nearly
all of them. So each column deserves an **embedding table over its exact values** — a learned
lookup table, which is literally the structure the generator created.

**Finding 2 — there is a hard generative constraint.**
`daily_screen_time_hours ≥ social + gaming + work` holds in **100.00000%** of all 859,029
train+test rows, minimum gap exactly 0.000, with 546 rows sitting on the boundary. So those
three columns are *components* of daily screen time and the remainder — unaccounted screen
time — is a real latent variable (standalone AUC 0.765). A GBDT cannot build a 4-term linear
combination out of axis-aligned splits, so it never finds this on its own.

**The architecture that follows from this:**

| component | why |
|---|---|
| Embedding table per column over exact values | captures the memorised lookup |
| Periodic-Linear (learned Fourier) numeric embedding, *added* to it | captures the smooth trend |
| NaN = index 0 → a **learned** per-column vector | beats imputation, which measures as useless here |
| Derived budget tokens | injects the constraint |
| Attention over feature tokens | represents the component interactions we measured |
| Random extra masking during training | augmentation over missingness patterns |

Measured OOF, 10-fold, identical splits across all models:

| model | OOF AUC |
|---|---|
| raw LightGBM (3-fold reference) | 0.96432 |
| LightGBM + value-level target encoding + constraint | 0.96824 |
| CatBoost + constraint | 0.96832 |
| **Lookup-Transformer** | **0.96872** |
| **blend (0.49 / 0.31 / 0.21)** | **0.96940** |

The network is both the strongest single model and far more *decorrelated* from the trees
(rank corr 0.968) than the trees are from each other (0.987), which is why it carries the
blend at roughly half the weight.

Every model writes its own `submission_<name>.csv` in addition to the blended
`submission.csv`, so each can be submitted and scored independently.


### このセルがやっていること（What）**設定と共通ユーティリティの定義**です。- ライブラリのimport（PyTorch、sklearn、`rankdata`）- `QUICK` フラグ（True にすると3fold・短時間の試運転、Falseで本番の11fold・32エポック）- `AMP_DT` の自動選択 — GPUが bfloat16 に対応していれば bf16、していなければ fp16- `save_model()` — 各モデルのOOF予測・テスト予測・**そのモデル単独の提出ファイル**を保存する関数### なぜそうするのか（Why）**理由1: `QUICK` フラグは実験の生命線。** 本番設定で1回30分かかるnotebookを、コードのバグ確認のために毎回フルで回すのは時間の無駄です。「3foldで数分」の高速モードを最初から用意しておくと、開発サイクルが劇的に速くなります。**強く真似すべき習慣**。**理由2: bf16とfp16の使い分け。** どちらも16bitで計算して速度とメモリを稼ぐ **AMP（混合精度学習）** の設定ですが、- **bfloat16** は指数部がfloat32と同じ幅なので**オーバーフロー／アンダーフローに強く、GradScalerが不要**- **float16** は精度は高いが表現範囲が狭く、勾配が0に潰れるのを防ぐ **GradScaler** が必要T4/P100（古いGPU）はbf16非対応なので、`torch.cuda.is_bf16_supported()` で分岐し、fp16のときだけ `USE_SCALER = True` にしています。この分岐を怠ると、環境によって学習が発散します。**理由3: モデルごとに提出ファイルを保存する。** 後でブレンドするだけでなく、**各モデル単独のLBスコアを確認できる**ようにしています。「OOFでは良いのにLBでは悪い」モデルを特定でき、ブレンド重みの妥当性を検証できます。> 補足: **OOF（Out-Of-Fold）予測** とは、交差検証で「その行が検証側に回ったときの予測値」を全行分集めたもの。> 学習に使っていないデータでの予測なので、**ブレンド重みの推定やモデル比較に安全に使えます**。> **`allow_tf32 = True`** はAmpere以降のGPUで行列積を高速化する設定です。

In [ ]:
import os, math, time, itertools
import numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import QuantileTransformer
from scipy.stats import rankdata

IN = '/kaggle/input/competitions/playground-series-s6e8'
WORK = '/kaggle/working'

QUICK    = False   # True -> 3 folds and shorter training, for a fast dry run
N_FOLDS  = 3 if QUICK else 11
EPOCHS   = 16 if QUICK else 32
SEED     = 42
RUN_GBDT = True    # CatBoost + LightGBM companions for the blend

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cuda.matmul.allow_tf32 = True
# T4/P100 do not support bf16; Ampere+ does. Pick the right autocast dtype.
AMP_DT = torch.bfloat16 if (dev == 'cuda' and torch.cuda.is_bf16_supported()) else torch.float16
USE_SCALER = (AMP_DT == torch.float16)
print('device', dev, '| amp', AMP_DT, '| folds', N_FOLDS)


SUBS = {}   # name -> (oof AUC, submission path)

def save_model(name, oof, test_pred):
    """Persist one model's OOF, test predictions and its OWN submission file."""
    auc = roc_auc_score(y, oof)
    np.save(f'{WORK}/oof_{name}.npy', oof)
    np.save(f'{WORK}/test_{name}.npy', test_pred)
    p = rankdata(test_pred) / len(test_pred)          # rank-normalise: AUC is rank-based
    path = f'{WORK}/submission_{name}.csv'
    pd.DataFrame({'id': test.id, 'addicted_label': p}).to_csv(path, index=False)
    SUBS[name] = (auc, path)
    print(f'[{name}] OOF AUC = {auc:.5f}  ->  {path}')
    return auc


## Data


### このセルがやっていること（What）train.csv / test.csv を読み込み、特徴量列 `FE`、目的変数 `y` を用意し、**trainとtestを縦に連結した `both`** を作ります。最後に基準率（陽性率）と列ごとの欠損率を表示。### なぜそうするのか（Why）**`both` を作るのがこの notebook の要です。** 発見1の「値をルックアップキーとして使う」戦略では、**train と test に出てくる全ての値を通し番号（ID）に変換**する必要があります。trainだけで語彙を作ると、testにしか出てこない値が「未知」になって情報を捨てることになります。ここで「それはリークでは？」と思うのが正しい感覚ですが、**これはリークではありません**。語彙を作る操作も、後で出てくる rank-gauss 変換も、**目的変数 `y` を一切参照していない**からです。このように「特徴量の分布だけを使う前処理」を train+test 全体で行うのは**transductive（トランスダクティブ）な前処理**と呼ばれ、Kaggleでは一般に許容されます。一方、**目的変数を使う前処理（target encoding など）を train+test でやるのは本物のリーク**です。この notebook は後で target encoding も使いますが、そちらは必ず fold 内に閉じて計算しています（後述）。この線引きが重要。> 補足: **基準率（base rate）** は陽性クラスの割合。不均衡の度合いを最初に確認するのは基本動作です。

In [ ]:
train = pd.read_csv(f'{IN}/train.csv')
test  = pd.read_csv(f'{IN}/test.csv')
FE = [c for c in test.columns if c != 'id']
y = train.addicted_label.values.astype(np.float32)
NTR, NTE = len(train), len(test)
both = pd.concat([train[FE], test[FE]], ignore_index=True)
print(train.shape, test.shape, '| base rate %.4f' % y.mean())
print('missing per column:')
print(both.isna().mean().round(4).to_string())


## Finding 1 — the exact value is a lookup key

The decisive test, model-free with respect to the claim: fit a model on half the data,
score the other half, split *that* half in two, and correlate the per-value mean residuals
of the two independent quarters. Composition effects are already absorbed by the model and
the two quarters' noise is independent, so under the null this correlation is 0.

*(Set `RUN_PROOF = True` to reproduce — it trains one extra LightGBM, ~2 min.)*


### このセルがやっていること（What）**発見1を証明するための実験コード**です（`RUN_PROOF = False` なので既定では実行されません）。手順:1. trainをランダムに半分（A・B）に分ける2. A で LightGBM を学習し、**B を予測して残差** `y - p` を得る3. B をさらに2つの四半分（q1・q2）に分ける4. 各列について「**同じ値を持つ行の残差の平均**」を q1・q2 それぞれで計算する5. その2つの相関を取る### なぜそうするのか（Why）**この実験設計が本当に美しいので、じっくり読む価値があります。**素朴に「値ごとのラベル平均」を見ると、**構成効果（composition effect）**に騙されます。たとえば「`notifications_per_day = 47` の人はたまたま年齢が若い人が多い」だけかもしれず、それは値そのものの効果ではありません。そこでまず**モデルを噛ませて残差を取る**。モデルが説明できる分（他の列との相関、構成効果）は残差から除かれるので、**残った残差は「モデルが説明できなかった、値そのものに紐づく成分」**になります。さらに巧妙なのが **2つの独立した四半分で相関を取る**という部分。もし値ごとの残差がただのノイズなら、q1のノイズとq2のノイズは無相関なので**相関は0**になるはずです（帰無仮説）。それが 0.72 も出るということは、**両方の四半分に共通して存在する、実在する信号**だという強い証拠です。これは統計学でいう **split-half reliability（折半法信頼性）** の応用で、「ノイズと本物のシグナルを分離する」古典的かつ強力な手法です。`(g.n1 >= 150) & (g.n2 >= 150)` というフィルタも重要で、**サンプル数が少ない値を除外**しないと、少数サンプル由来の偶然の一致が相関を膨らませてしまいます。> 補足: **残差（residual）** = 実際の値 − 予測値。「モデルがまだ捉えきれていない部分」を表します。> **t統計量** も併記されており、相関が偶然で説明できるかを定量的に判断しています。

In [ ]:
RUN_PROOF = False

if RUN_PROOF:
    import lightgbm as lgb
    tp = train.copy()
    for c in [c for c in FE if tp[c].dtype == object]:
        tp[c] = tp[c].astype('category')
    rng = np.random.default_rng(0)
    perm = rng.permutation(NTR)
    A, B = perm[:NTR // 2], perm[NTR // 2:]
    m = lgb.LGBMClassifier(n_estimators=1500, learning_rate=0.03, num_leaves=127,
                           colsample_bytree=0.8, subsample=0.8, subsample_freq=1,
                           min_child_samples=60, verbose=-1)
    m.fit(tp[FE].iloc[A], y[A])
    pB = m.predict_proba(tp[FE].iloc[B])[:, 1]
    res = pd.DataFrame({'r': y[B] - pB, 'half': np.where(np.arange(len(B)) < len(B) // 2, 1, 2)})
    for c in FE:
        res[c] = train[c].iloc[B].astype(str).values
    print(f'{"column":26s} {"corr(q1,q2)":>12s} {"t":>8s}')
    for c in FE:
        g = res.groupby([c, 'half']).r.agg(['mean', 'size']).unstack('half')
        g.columns = ['d1', 'd2', 'n1', 'n2']
        g = g.dropna()
        g = g[(g.n1 >= 150) & (g.n2 >= 150)]
        if len(g) < 8:
            continue
        r = np.corrcoef(g.d1, g.d2)[0, 1]
        t = r * np.sqrt(len(g) - 2) / np.sqrt(max(1 - r ** 2, 1e-9))
        print(f'{c:26s} {r:12.4f} {t:8.2f}')
else:
    print('Previously measured (t-stats, split-half reproducibility of per-value residuals):')
    print('  daily_screen_time_hours  corr 0.72  t 19.8')
    print('  weekend_screen_time      corr 0.70  t 18.8')
    print('  notifications_per_day    corr 0.72  t 15.4   <- univariate AUC is only 0.492')
    print('  app_opens_per_day        corr 0.68  t 11.8')
    print('  sleep_hours              corr 0.50  t 10.8')


## Finding 2 — the time-budget constraint


### このセルがやっていること（What）**発見2（時間予算制約）の検証**です。- `daily_screen_time_hours − (social + gaming + work)` を全行で計算- それが 0 以上である割合、最小ギャップ、ちょうど0の行数、中央値の余裕を表示- その差分（＝説明のつかないスクリーンタイム）の**単独AUC**を計算- **対照実験（control）** として、同じ関係が `weekend_screen_time` でも成り立つかを確認### なぜそうするのか（Why）**理由1: 「100.00000%」は偶然ではありえない。** 859,029行すべてで不等式が成立し、最小ギャップがぴったり 0.0000 で、境界上に546行が乗っている。これは**データ生成器が明示的にこの制約を課している**ことを意味します。実データでは絶対にこんな綺麗な関係は出ません。**理由2: 対照実験があるから主張が強い。** ここが特に良い部分です。「weekend にも同じ制約があるか？」を調べて、**ないことを確認**している。もし全部の列ペアでこういう関係が出るなら、それはデータの性質ではなく検証方法の不備です。**成立しない例を示すことで、成立する例の意味が確定する**——これが対照実験の役割です。**理由3: なぜGBDTでは足りないのか。** 決定木は「1つの特徴量を1つの閾値で分割」しか作れません。`a − b − c − d` という4項の引き算は、木を無限に深くしても近似しかできない。だから**人間が明示的に計算して特徴量として渡す**必要があります。「木が苦手な形の関係を見つけて手で作ってやる」のは、テーブルコンペの特徴量エンジニアリングの本質そのものです。> 補足: **単独AUC 0.765** は非常に強い数字です。1つの派生特徴量だけでこれだけ判別できるということは、> この潜在変数がラベル生成に直接使われている可能性が高い、ということを示唆します。

In [ ]:
COMP = ['social_media_hours', 'gaming_hours', 'work_study_hours']
d = (both.daily_screen_time_hours - both[COMP].sum(1)).dropna()
print('daily >= social+gaming+work')
print('  holds in %.5f%% of %d rows' % (100 * (d >= -1e-9).mean(), len(d)))
print('  min gap %.4f   exact zeros %d   median slack %.2f h' % (d.min(), (d.abs() < 1e-9).sum(), d.median()))
print()
ok = d.notna()
print('standalone AUC of the unaccounted-screen-time remainder: %.4f'
      % roc_auc_score(y[d.index[d.index < NTR]], d[d.index < NTR]))
print()
print('weekend has NO such constraint (control):')
dw = (both.weekend_screen_time - both[['social_media_hours', 'gaming_hours']].sum(1)).dropna()
print('  weekend >= social+gaming holds only %.5f%%, min gap %.2f' % (100 * (dw >= -1e-9).mean(), dw.min()))


## Feature preparation

Three blocks: integer ids for the lookup tables (0 = NaN), rank-gauss numerics for the
smooth branch, and the derived budget quantities. Nothing here touches the target, so all
of it may be fit on train+test — that is not leakage.


### このセルがやっていること（What）**3種類の特徴量ブロック**を作ります。1. **ルックアップID** — 各列の値を文字列化し、辞書順に通し番号を振る。`0` は欠損（`__NA__`）専用に予約。   `OFF`（オフセット）で各列のID空間をずらし、**全列で1つの巨大な埋め込みテーブルを共有**できるようにする。2. **派生した予算特徴量** — `other_screen`（余り）、`sgw`（3つの合計）、`other_frac`（余りの割合）など6個。3. **rank-gauss 変換** — 数値列を順位に直してから正規分布の分位点に写す。### なぜそうするのか（Why）**理由1: 欠損に 0 を予約する意味。** 欠損を平均値などで補完せず、**専用のIDを与えて埋め込みを学習させる**。これにより「欠損していること自体が持つ情報」をモデルが自分で学べます。このデータでは61%の行に欠損があるので、これは大きな設計判断です。**理由2: オフセットで埋め込みを1本化する理由。** 列ごとに別々の `nn.Embedding` を持つより、1つの大きなテーブルにオフセットでアクセスするほうが、GPU上のメモリ配置と計算がずっと効率的です。`OFF = cumsum(vocab)` で各列の開始位置を計算し、`ID + OFF[列]` でアクセスします。実装上の定石。**理由3: rank-gauss 変換の効果。** 生の値には外れ値や歪んだ分布がありますが、順位に変換してから正規分布に写すと、**必ず綺麗な正規分布**になります。ニューラルネットは入力が正規化されているほど学習が安定する（勾配が飛びにくい）ので、テーブルデータ×NN では標準化や min-max より rank-gauss が好まれます。**理由4: `clip(0.1)` の意味。** `other_frac` は割り算なので、分母が0に近いと値が発散します。`clip(0.1)` で下限を設けて**ゼロ除算と極端な値を防いで**います。地味ですが必須のガードです。**なぜ train+test 全体で計算してよいのか**: 前述のとおり、この3ブロックはどれも `y` を参照していません。原著者もコメントで「Nothing here touches the target, so all of it may be fit on train+test — that is not leakage.」と明言しています。> 補足: **語彙（vocabulary）** は自然言語処理の用語で「扱う単語の集合」。> ここでは「各列に出現する値の集合」を語彙として扱っている——**数値をトークンとして扱う**という発想の転換です。

In [ ]:
# ---- lookup ids: exact value -> int, 0 reserved for NaN ----
ids, vocab = [], []
for c in FE:
    s = both[c].astype(str).where(both[c].notna(), '__NA__')
    cats = sorted(v for v in s.unique() if v != '__NA__')
    mp = {v: i + 1 for i, v in enumerate(cats)}
    mp['__NA__'] = 0
    ids.append(s.map(mp).values.astype(np.int64))
    vocab.append(len(cats) + 1)
IDS = np.stack(ids, 1)
OFF = np.concatenate([[0], np.cumsum(vocab)[:-1]]).astype(np.int64)
TOTV = int(sum(vocab))
print('vocab per column:', dict(zip(FE, vocab)))

# ---- derived budget tokens ----
sgw = both[COMP].sum(1)
DERIV = pd.DataFrame({
    'other_screen': both.daily_screen_time_hours - sgw,
    'sgw': sgw,
    'other_frac': (both.daily_screen_time_hours - sgw) / both.daily_screen_time_hours.clip(0.1),
    'wk_minus_sgw': both.weekend_screen_time - sgw,
    'wk_other': both.weekend_screen_time - (both.daily_screen_time_hours - sgw),
    'sgw_frac': sgw / both.daily_screen_time_hours.clip(0.1),
})

def rank_gauss(df):
    X = np.zeros((len(df), df.shape[1]), np.float32)
    M = np.zeros_like(X)
    for j, c in enumerate(df.columns):
        v = df[c].values.astype(np.float64)
        o = ~np.isnan(v)
        if o.sum() > 10:
            q = QuantileTransformer(n_quantiles=1000, output_distribution='normal',
                                    subsample=400000, random_state=0)
            X[o, j] = q.fit_transform(v[o].reshape(-1, 1)).ravel().astype(np.float32)
        M[~o, j] = 1.0
    return X, M

RAWNUM = pd.DataFrame({c: (both[c] if both[c].dtype != object else np.nan) for c in FE})
CN, CM = rank_gauss(RAWNUM)   # smooth branch for the 12 raw columns
DN, DM = rank_gauss(DERIV)    # derived budget tokens
NCAT, NDER = len(FE), DERIV.shape[1]
print(f'tokens: CLS + {NCAT} lookup + {NDER} derived = {1 + NCAT + NDER}')

t_ids = torch.from_numpy(IDS + OFF[None, :])
t_cn, t_cm = torch.from_numpy(CN), torch.from_numpy(CM)
t_dn, t_dm = torch.from_numpy(DN), torch.from_numpy(DM)
t_y = torch.from_numpy(y)


## The model

Each raw column contributes one token:

```
token_j = Embedding_j[exact_value_id]  +  PLR_j(rank_gauss_value) * (1 - is_missing)
```

The embedding is the learned lookup table; PLR is a **Periodic-Linear** encoder with learned
Fourier frequencies, which represents smooth trends far more efficiently than feeding a raw
scalar into an MLP. When a value is missing the smooth term is switched off and only
`Embedding_j[0]` remains — so *missing* has its own learned representation per column, rather
than being imputed. Derived budget features get PLR tokens only. A CLS token reads out.


### このセルがやっていること（What）**モデル本体の定義**です。2つのクラスがあります。**`PLR`（Periodic-Linear Representation）** — 数値を埋め込みに変換するモジュール。学習可能な周波数 `f` を使って `z = 2π · x · f` を作り、`sin(z)` と `cos(z)` を連結してから線形変換します。**`LookupTransformer`** — 全体のモデル。- `self.emb`: 全列の値を1本にまとめた埋め込みテーブル（＝**学習可能なルックアップ表**）- `plr_c` / `plr_d`: 生の列用・派生列用のPLR- `cls`: 全体の要約を読み出すためのCLSトークン- `pos`: 位置埋め込み（どのトークンがどの列かを区別する）- `TransformerEncoderLayer` を4層、8ヘッド、GELU活性化### なぜそうするのか（Why）**理由1: なぜ sin/cos で展開するのか。** ニューラルネットに生のスカラー `x` を1本だけ入れると、最初の線形層は `wx + b` という直線しか作れず、細かい非線形性を表現するのに多くの層が必要です。`sin(2πfx)`, `cos(2πfx)` を複数の周波数 `f` で並べると、**フーリエ級数のように任意の滑らかな関数を効率よく近似**できます。しかも周波数 `f` 自体が学習可能なので、**そのデータに必要な解像度をモデルが自分で選べる**。NeRF（3D再構成）の positional encoding と同じ発想で、近年テーブルデータ×NNの標準的な武器になっています。**理由2: 埋め込みとPLRを「足す」意味。**```token = Embedding[厳密な値] + PLR(滑らかな値) × (1 - 欠損)```埋め込みは**離散的な暗記**（発見1）、PLRは**連続的な傾向**を担当します。足し算にすることで、モデルは「この列は暗記が効く」「この列は滑らかな傾向が主」を**列ごとに自動で使い分け**られます。片方だけでは取れない情報を両方カバーする、非常にうまい設計です。**理由3: 欠損時に `(1 - is_missing)` を掛ける意味。** 欠損なら滑らかな項が消え、`Embedding[0]`（欠損専用の埋め込み）だけが残ります。**補完せずに欠損を表現する**という一貫した思想です。**理由4: なぜTransformerか。** 各列を独立に処理するのではなく、**self-attentionで列同士の相互作用を学習**できます。「スクリーンタイムが長い**かつ**年齢が若い」のような組み合わせ効果を、手で交互作用項を作らずに学べます。**理由5: 埋め込みの初期化 `std=0.02`。** 大きすぎると学習初期に発散し、小さすぎると勾配が流れません。0.02 はBERT以来のTransformer系で広く使われる経験的な値です。> 補足: **CLSトークン** はBERT由来の手法。系列に「要約用の特別なトークン」を1つ足しておき、> attentionで全体の情報を集約させ、最後にそのトークンの出力だけを分類器に渡します。

In [ ]:
class PLR(nn.Module):
    """Periodic-Linear numeric embedding (learned Fourier frequencies per feature)."""
    def __init__(self, nfeat, k, d, sigma=0.5):
        super().__init__()
        self.f = nn.Parameter(torch.randn(nfeat, k) * sigma)
        self.w = nn.Parameter(torch.randn(nfeat, 2 * k, d) / math.sqrt(2 * k))
        self.b = nn.Parameter(torch.zeros(nfeat, d))
    def forward(self, x):
        z = 2 * math.pi * x.unsqueeze(-1) * self.f.unsqueeze(0)
        z = torch.cat([torch.sin(z), torch.cos(z)], -1)
        return torch.einsum('bfk,fkd->bfd', z, self.w) + self.b


class LookupTransformer(nn.Module):
    def __init__(self, totv, ncat, nder, d=128, k=24, layers=4, heads=8, drop=0.1):
        super().__init__()
        self.emb = nn.Embedding(totv, d)          # <- the learned lookup tables
        nn.init.normal_(self.emb.weight, std=0.02)
        self.plr_c = PLR(ncat, k, d)
        self.plr_d = PLR(nder, k, d)
        self.cls = nn.Parameter(torch.zeros(1, 1, d))
        self.pos = nn.Parameter(torch.randn(1, 1 + ncat + nder, d) * 0.02)
        self.edrop = nn.Dropout(drop)
        enc = nn.TransformerEncoderLayer(d, heads, d * 2, drop, activation='gelu',
                                         batch_first=True, norm_first=True)
        self.tr = nn.TransformerEncoder(enc, layers)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, d), nn.GELU(),
                                  nn.Dropout(drop), nn.Linear(d, 1))
    def forward(self, idx, cn, cm, dn, dm):
        B = idx.shape[0]
        tok_c = self.emb(idx) + self.plr_c(cn) * (1 - cm).unsqueeze(-1)
        tok_d = self.plr_d(dn) * (1 - dm).unsqueeze(-1)
        t = torch.cat([self.cls.expand(B, -1, -1), tok_c, tok_d], 1) + self.pos
        return self.head(self.tr(self.edrop(t))[:, 0]).squeeze(-1)

print(LookupTransformer(TOTV, NCAT, NDER))


## Training

AdamW with a larger weight decay on the embedding tables than on everything else (they are
the part that overfits), OneCycle schedule, EMA weights, and random extra masking as
augmentation. Validation AUC is evaluated on the EMA copy.


### このセルがやっていること（What）**1つのfoldを学習する関数 `run_fold`** です。含まれる要素:- **AdamW**、ただし**埋め込みだけ weight decay を強く**（`3e-4` vs 他 `1e-5`）- **OneCycleLR** スケジュール（`pct_start=0.15`）- **EMA（指数移動平均）** の重みを別に保持し、**検証はEMA側で行う**- **ランダムな追加マスキング**（`aug=0.10`）をデータ拡張として使用- データを事前にGPUへ転送、チャンク単位で推論### なぜそうするのか（Why）**理由1: 埋め込みだけ強く正則化する理由。** これが最重要ポイントです。埋め込みテーブルは「値ごとに独立したパラメータ」なので、**最も過学習しやすい部分**です。サンプル数の少ない値の埋め込みは、その数行のノイズを丸暗記してしまう。weight decay を30倍強くすることで、データが少ない値の埋め込みは0に引き戻され、**十分なデータがある値だけが独自の表現を保持**します。これは実質的に **target encoding の平滑化（smoothing）と同じ効果**を、勾配降下で自動的に実現しています。**理由2: OneCycleLR。** 学習率を「低→高→低」と1周させるスケジュール。序盤に上げることで局所解から抜け出しやすくし、終盤に下げて丁寧に収束させます。`pct_start=0.15` は「最初の15%で上げきる」設定。固定学習率より速く、かつ良い解に到達しやすい。**理由3: EMA（Exponential Moving Average）。** 学習中の重みの移動平均を別に保持します。SGD系の最適化は最適解の周りを振動するので、その**平均を取ると振動が打ち消され**、汎化性能が上がることが多い。「タダで手に入る軽量なアンサンブル」と考えると分かりやすい。**検証をEMA側で行っている**のが正しい使い方です（本番で使う重みで選ぶ）。**理由4: ランダムマスキングをデータ拡張に使う理由。** 学習時に10%の確率で特徴量を「欠損」に置き換えます。これはNLPのBERTのマスク学習と同じ発想で、**モデルが1つの列に依存しすぎるのを防ぎ**、残りの列から推論する力を鍛えます。このデータは元々61%の行に欠損があるので、**テスト時の分布にも近づく**という副次効果もあります。**理由5: データを丸ごとGPUに載せる理由。** DataLoaderでCPU→GPU転送を毎バッチ行うとそこがボトルネックになります。このデータはGPUメモリに収まるサイズなので、最初に一括転送してしまうほうが圧倒的に速い。> 補足: **weight decay** はパラメータを0に引き寄せる正則化（L2正則化に相当）。> **AdamW** は weight decay を勾配とは独立に適用する改良版で、現在のTransformer学習の標準です。

In [ ]:
def run_fold(trn, val, fold, epochs=EPOCHS, bs=2048, lr=2e-3, aug=0.10):
    torch.manual_seed(fold)
    g = torch.Generator().manual_seed(fold)
    model = LookupTransformer(TOTV, NCAT, NDER).to(dev)
    embp = [p for n, p in model.named_parameters() if n.startswith('emb')]
    rest = [p for n, p in model.named_parameters() if not n.startswith('emb')]
    opt = torch.optim.AdamW([{'params': rest, 'weight_decay': 1e-5},
                             {'params': embp, 'weight_decay': 3e-4}], lr=lr)
    nstep = math.ceil(len(trn) / bs) * epochs + 10
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=nstep, pct_start=0.15)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_SCALER)
    lossf = nn.BCEWithLogitsLoss()
    P = list(model.parameters())
    ema = [p.detach().clone() for p in P]

    I  = t_ids[trn].to(dev); CNt = t_cn[trn].to(dev); CMt = t_cm[trn].to(dev)
    DNt = t_dn[trn].to(dev); DMt = t_dm[trn].to(dev); Y = t_y[trn].to(dev)
    Iv = t_ids[val].to(dev); CNv = t_cn[val].to(dev); CMv = t_cm[val].to(dev)
    DNv = t_dn[val].to(dev); DMv = t_dm[val].to(dev)
    OFFT = torch.from_numpy(OFF).to(dev)

    def predict(a, c, cm, dd, dm, chunk=16384):
        model.eval()
        with torch.no_grad(), torch.autocast('cuda', dtype=AMP_DT):
            return torch.cat([model(a[i:i+chunk], c[i:i+chunk], cm[i:i+chunk],
                                    dd[i:i+chunk], dm[i:i+chunk])
                              for i in range(0, len(a), chunk)]).float().cpu().numpy()

    best, best_w, bad = 0.0, None, 0
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(trn), generator=g).to(dev)
        for i in range(0, len(trn), bs):
            sl = perm[i:i+bs]
            idx, cm = I[sl].clone(), CMt[sl].clone()
            if aug > 0:   # hide extra values -> learn every missingness pattern
                drop = torch.rand(idx.shape, device=dev) < aug
                idx = torch.where(drop, OFFT.expand_as(idx), idx)
                cm = torch.maximum(cm, drop.float())
            with torch.autocast('cuda', dtype=AMP_DT):
                loss = lossf(model(idx, CNt[sl], cm, DNt[sl], DMt[sl]), Y[sl])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(P, 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            with torch.no_grad():
                torch._foreach_mul_(ema, 0.999)
                torch._foreach_add_(ema, [p.detach() for p in P], alpha=0.001)
        if ep >= 5 and (ep % 2 == 1 or ep == epochs - 1):
            bak = [p.detach().clone() for p in P]
            with torch.no_grad():
                for p, e in zip(P, ema): p.copy_(e)
            a = roc_auc_score(y[val], predict(Iv, CNv, CMv, DNv, DMv))
            if a > best:
                best, best_w, bad = a, [e.detach().clone() for e in ema], 0
            else:
                bad += 1
            print(f'    fold{fold} ep{ep:3d} valAUC={a:.5f} best={best:.5f}', flush=True)
            with torch.no_grad():
                for p, k in zip(P, bak): p.copy_(k)
            if bad >= 5: break
    with torch.no_grad():
        for p, e in zip(P, best_w): p.copy_(e)
    pv = predict(Iv, CNv, CMv, DNv, DMv)
    pt = predict(t_ids[NTR:].to(dev), t_cn[NTR:].to(dev), t_cm[NTR:].to(dev),
                 t_dn[NTR:].to(dev), t_dm[NTR:].to(dev))
    del I, CNt, CMt, DNt, DMt, Y, Iv, CNv, CMv, DNv, DMv
    torch.cuda.empty_cache()
    return pv, pt, best


### このセルがやっていること（What）**11-fold の StratifiedKFold で交差検証を回す**メインループです。各foldで `run_fold` を呼び、検証部分の予測を `oof_dl` に書き込み、テスト予測を11回分平均します。最後に `save_model` でOOF AUCを計算・保存。### なぜそうするのか（Why）**理由1: StratifiedKFold を使う理由。** 通常の KFold だと、fold ごとに陽性率が偏る可能性があります。Stratified は**各foldの陽性/陰性比率を全体と同じに保つ**ので、fold間のスコアのばらつきが小さくなり、CVスコアが信頼できる推定値になります。二値分類では基本的に常にこちらを使うべきです。**理由2: なぜ11 fold なのか。** fold数を増やすと各モデルの学習データが増え（11foldなら約91%）、性能が上がりやすくなります。また**奇数**にしておくと、後で使うかもしれない多数決系の処理で同点が起きません。デメリットは学習時間が11倍になること。原著者は `QUICK` で3foldに落とせるようにしてバランスを取っています。**理由3: テスト予測を「平均」する意味。** 11個のモデルはそれぞれ違うデータで学習しているので、それぞれ違う誤りをします。平均すると**誤りが打ち消し合い、分散が減る**——これが **bagging（バギング）** の効果で、交差検証をそのままアンサンブルとして使う定番テクニックです。**理由4: OOFを埋める意味。** `oof_dl[val] = pv` で、各行が検証側に回ったときの予測を記録します。全foldが終わると**全行について「学習に使っていないモデルの予測」**が揃い、これがブレンド重み探索の土台になります。> 補足: 同じ `FOLDS` 変数を後のGBDTでも使い回している点に注目。> **全モデルが同一のfold分割を共有する**ことで、OOF同士を直接比較・ブレンドできます。ここを揃えないと重み推定が壊れます。

In [ ]:
FOLDS = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(NTR), y))
oof_dl = np.zeros(NTR); test_dl = np.zeros(NTE)
t0 = time.time()
for f, (trn, val) in enumerate(FOLDS):
    pv, pt, bst = run_fold(trn, val, f)
    oof_dl[val] = pv; test_dl += pt / N_FOLDS
    print(f'  fold {f} done AUC={bst:.5f}  ({time.time()-t0:.0f}s)', flush=True)

AUC_DL = save_model('lookup_transformer', oof_dl, test_dl)


## GBDT companions

Two tree models for the blend. Both get the budget features; CatBoost additionally sees every
column *twice* — once numeric, once categorical — so its ordered target statistics encode the
lookup structure. LightGBM gets explicit fold-safe value-level target encoding instead.


### このセルがやっていること（What）**CatBoost の相棒モデル**です。特徴的なのは、**全ての列を2回ずつ**渡していること。- 1回目: `Xc[c] = both[c].astype(np.float32)` — **数値として**- 2回目: `Xc['k_' + c] = both[c].astype(str)` — **カテゴリとして**（`CATCOLS` に登録）さらに派生予算特徴量、`slack`（余り）、`n_comp_obs`（構成要素のうち観測されている数）を追加。`max_ctr_complexity=2`、`od_type='Iter'` で早期打ち切り。### なぜそうするのか（Why）**理由1: 同じ列を数値とカテゴリで二度渡す意味。** これが発見1をGBDTで再現するトリックです。- **数値として**渡せば、木は「> 47.5」のような**大小関係（単調な傾向）**を学べる- **カテゴリとして**渡せば、CatBoostの **ordered target statistics** が  「値47.00に対応するラベル平均」という**値そのものへの紐付け**を学べるつまり Lookup-Transformer の `Embedding + PLR` という二本立てを、**CatBoostの機能だけで擬似的に再現**しています。同じ思想の別実装。**理由2: CatBoostのordered target statisticsが優秀な理由。** 普通のtarget encodingは「その行自身のラベル」が encoding に入り込んでリークします。CatBoostは行をランダムに並べ、**各行について「それより前の行だけ」から統計量を計算**することで、原理的にリークを防ぎます。だから安心して高カーディナリティのカテゴリを渡せる。**理由3: `max_ctr_complexity=2` の意味。** CatBoostはカテゴリの**組み合わせ**からも特徴量を作りますが、列数が多いと組み合わせが爆発します。2に制限して計算量と過学習を抑えています。**理由4: `n_comp_obs` を入れる意味。** 「3つの構成要素のうち何個が観測されているか」。欠損の個数が多いほど `slack` の計算が信用できないので、**モデルに `slack` をどれくらい信じるべきかのヒント**を与えていることになります。うまい。> 補足: **早期打ち切り（early stopping）**: `od_wait=200` は「検証スコアが200回連続で改善しなければ止める」設定。> 6000本の木を指定していますが、実際には必要な本数で自動的に止まります。過学習防止と時間節約を同時に達成する定番設定。

In [ ]:
if RUN_GBDT:
    from catboost import CatBoostClassifier, Pool
    NUMC = [c for c in FE if train[c].dtype != object]
    Xc = pd.DataFrame(index=both.index)
    for c in NUMC:
        Xc[c] = both[c].astype(np.float32)
    for c in DERIV.columns:
        Xc[c] = DERIV[c].astype(np.float32)
    Xc['slack'] = (both.daily_screen_time_hours - both[COMP].fillna(0).sum(1)).astype(np.float32)
    Xc['n_comp_obs'] = both[COMP].notna().sum(1).astype(np.float32)
    for c in FE:
        Xc['k_' + c] = both[c].astype(str).fillna('nan')
    CATCOLS = ['k_' + c for c in FE]
    Xca, Xct = Xc.iloc[:NTR], Xc.iloc[NTR:]

    oof_cb = np.zeros(NTR); test_cb = np.zeros(NTE)
    for f, (trn, val) in enumerate(FOLDS):
        m = CatBoostClassifier(iterations=6000, learning_rate=0.05, depth=8, l2_leaf_reg=6,
                               eval_metric='AUC', random_seed=0, od_type='Iter', od_wait=200,
                               verbose=0, one_hot_max_size=4, max_ctr_complexity=2,
                               task_type='GPU' if dev == 'cuda' else 'CPU')
        m.fit(Pool(Xca.iloc[trn], y[trn], cat_features=CATCOLS),
              eval_set=Pool(Xca.iloc[val], y[val], cat_features=CATCOLS), use_best_model=True)
        oof_cb[val] = m.predict_proba(Pool(Xca.iloc[val], cat_features=CATCOLS))[:, 1]
        test_cb += m.predict_proba(Pool(Xct, cat_features=CATCOLS))[:, 1] / N_FOLDS
        print(f'  CB fold {f} AUC={roc_auc_score(y[val], oof_cb[val]):.5f}', flush=True)
    AUC_CB = save_model('catboost', oof_cb, test_cb)


### このセルがやっていること（What）**LightGBM の相棒モデル**です。CatBoostと違い、**target encoding を自前で、fold安全に**実装しています。- `cnt_列名`: 各値の**出現回数**（count encoding）- `te_cols()`: 平滑化つき target encoding を**入れ子の交差検証（inner KFold）**で計算平滑化の式:```encoding = (陽性数 + 全体平均 × α) / (件数 + α),  α = 30```### なぜそうするのか（Why）**理由1: 平滑化（smoothing）が必須な理由。** ある値がデータ中に3行しかなく、たまたま3行とも陽性だったとします。素朴なtarget encodingは 1.0 を返しますが、これはほぼ確実にノイズです。α=30 の平滑化を入れると `(3 + 0.5×30)/(3 + 30) ≈ 0.545` となり、**全体平均に強く引き戻されます**。逆に1000行ある値なら α の影響は小さく、実測値がほぼそのまま使われる。**「データが少ないほど事前分布を信じる」というベイズ的な収縮**そのものです。前セルで埋め込みに強い weight decay をかけたのと**まったく同じ思想**である点に注目してください。**理由2: 入れ子の交差検証（nested CV）が必須な理由。** ここが最も間違えやすい部分です。target encoding をfold内の全学習データで計算し、そのままそのfoldの学習に使うと、**学習データの各行が自分自身のラベルで作られた特徴量を見る**ことになり、深刻なリークが起きます。CVスコアだけ跳ね上がってLBで惨敗する典型パターンです。対策として、学習データをさらに5分割し、`ctr[i2] = 内側fold i1 から作った encoding` として**その行を含まないデータから計算**しています。検証データとテストデータには、学習データ全体（`full`）から作った encoding を適用。この「**内側は入れ子、外側は全体**」というパターンは覚えておく価値があります。**理由3: count encoding を入れる理由。** 「この値が何回出現するか」は、それ自体が信号になることがあります（レアな値 vs よくある値）。また、モデルが「この値の target encoding をどれくらい信じるべきか」を判断する材料にもなります。**理由4: LightGBMとCatBoostを両方使う理由。** 同じGBDTでもカテゴリの扱い方が根本的に違うため、**予測の誤りパターンが異なります**。誤りが異なるモデルほどブレンドの効果が大きい。> 補足: `fillna(PRIOR)` — 検証/テストにしか出てこない未知の値には全体平均を割り当てます。> 未知値への安全なフォールバックとして必須です。

In [ ]:
if RUN_GBDT:
    import lightgbm as lgb
    PRIOR, ALPHA = float(y.mean()), 30.0
    keys = {c: both[c].astype(str) for c in FE}
    Xl = both.copy()
    for c in [c for c in FE if Xl[c].dtype == object]:
        Xl[c] = Xl[c].astype('category')
    for c in FE:
        Xl['cnt_' + c] = keys[c].map(keys[c].value_counts()).values.astype(np.float32)
    for c in DERIV.columns:
        Xl[c] = DERIV[c].astype(np.float32)
    Xla = Xl.iloc[:NTR].reset_index(drop=True); Xlt = Xl.iloc[NTR:].reset_index(drop=True)

    def te_cols(k_all, trn, val, alpha=ALPHA):
        k = k_all[:NTR]
        inner = list(StratifiedKFold(5, shuffle=True, random_state=7).split(trn, y[trn]))
        def mp(idx):
            g = pd.DataFrame({'k': k[idx], 'y': y[idx]}).groupby('k').y.agg(['sum', 'count'])
            return (g['sum'] + PRIOR * alpha) / (g['count'] + alpha)
        ctr = np.full(len(trn), PRIOR)
        for i1, i2 in inner:
            ctr[i2] = pd.Series(k[trn[i2]]).map(mp(trn[i1])).fillna(PRIOR).values
        full = mp(trn)
        return ctr, pd.Series(k[val]).map(full).fillna(PRIOR).values, \
               pd.Series(k_all[NTR:]).map(full).fillna(PRIOR).values

    oof_lgb = np.zeros(NTR); test_lgb = np.zeros(NTE)
    for f, (trn, val) in enumerate(FOLDS):
        Xtr, Xva, Xte = Xla.iloc[trn].reset_index(drop=True), Xla.iloc[val].reset_index(drop=True), Xlt.copy()
        for c in FE:
            a, b_, t_ = te_cols(keys[c].values, trn, val)
            Xtr['te_' + c], Xva['te_' + c], Xte['te_' + c] = a, b_, t_
        m = lgb.LGBMClassifier(n_estimators=5000, learning_rate=0.03, num_leaves=127,
                               colsample_bytree=0.8, subsample=0.8, subsample_freq=1,
                               min_child_samples=60, max_bin=2047, verbose=-1,
                               random_state=11, force_row_wise=True)
        m.fit(Xtr, y[trn], eval_set=[(Xva, y[val])], eval_metric='auc',
              callbacks=[lgb.early_stopping(150, verbose=False)])
        oof_lgb[val] = m.predict_proba(Xva)[:, 1]
        test_lgb += m.predict_proba(Xte)[:, 1] / N_FOLDS
        print(f'  LGB fold {f} AUC={roc_auc_score(y[val], oof_lgb[val]):.5f}', flush=True)
    AUC_LGB = save_model('lightgbm', oof_lgb, test_lgb)


## Blend and submit

Every model above has already written **its own `submission_<name>.csv`**, so each can be
submitted on its own — useful for checking how each one scores on the leaderboard
independently, and for reusing a single model without re-running the notebook.

This cell adds the blended `submission.csv`. Weights are searched on the OOF predictions;
all models share the same fold split, so the weights are estimated honestly.


### このセルがやっていること（What）**ブレンドと提出ファイルの作成**です。1. 各モデルのOOFとテスト予測を `rankdata(v)/len(v)` で**順位に正規化**2. 各モデル単独のOOF AUCを表示3. モデル**間の順位相関**を表示4. 重みを 0〜1 の 0.05 刻みで**総当たり探索**し、OOF AUCが最大になる組み合わせを選ぶ5. その重みでテスト予測をブレンドし、min-maxで[0,1]に正規化して `submission.csv` を書き出す### なぜそうするのか（Why）**理由1: なぜ順位に変換してから混ぜるのか。** 3つのモデルは出力のスケールが違います（NNは0付近に集中しがち、GBDTは広がりやすい、など）。生の確率を平均すると**スケールの大きいモデルの意見が不当に強くなります**。順位に直せば全モデルが一様分布[0,1]になり、公平に混ざる。**評価指標がAUC（順位しか見ない）なので、順位で混ぜるのは指標と完全に整合的**です。**理由2: 順位相関を表示する意味。** ここは見落とされがちですが重要です。2つのモデルの相関が 0.99 なら、混ぜてもほぼ何も変わりません。相関が 0.90 程度なら、**互いに違う誤りをしている**のでブレンドで大きく伸びます。「どのモデルを追加すべきか」を判断するための診断情報であり、**強いモデルを増やすより、相関の低いモデルを増やすほうが効く**というアンサンブルの鉄則を確認する手段です。**理由3: 総当たり探索が許される理由。** モデルが3つなら 21×21×21 ≈ 9,261通りで一瞬です。ロジスティック回帰などでスタッキングする方法もありますが、モデル数が少ないなら総当たりのほうが単純で、過学習も起きにくい。**理由4: 重みをOOFで探索するのが「正直」な理由。** 原著者が「all models share the same fold split, so the weights are estimated honestly」と書いているとおりです。**LB（公開スコア）を見ながら重みを手で調整すると、公開テストセットに過学習**します。Playgroundで上位が最終日に崩れる最大の原因がこれ。OOFで決めれば、private LBでも通用する重みになります。**理由5: min-max正規化してもAUCは変わらない。** 単調増加変換は順位を変えないため。提出値の見た目を整えているだけです。> 補足: **`rankdata`** は同順位に平均順位を割り当てます（1,2,2,4 ではなく 1,2.5,2.5,4）。> 同じ値のサンプルを恣意的に順序付けないための配慮です。

In [ ]:
O = {'lookup_transformer': oof_dl}; T = {'lookup_transformer': test_dl}
if RUN_GBDT:
    O['catboost'], T['catboost'] = oof_cb, test_cb
    O['lightgbm'], T['lightgbm'] = oof_lgb, test_lgb

R  = {k: rankdata(v) / len(v) for k, v in O.items()}
RT = {k: rankdata(v) / len(v) for k, v in T.items()}
for k, v in O.items():
    print(f'{k:20s} OOF AUC {roc_auc_score(y, v):.5f}')
print()
for a, b in itertools.combinations(R, 2):
    print(f'  rank corr {a:20s} vs {b:20s} = {np.corrcoef(R[a], R[b])[0,1]:.4f}')

ks = list(R)
best = (0, None)
if len(ks) > 1:
    for w in itertools.product(np.arange(0, 1.02, 0.05), repeat=len(ks)):
        s = sum(w)
        if s < 1e-9: continue
        ww = np.array(w) / s
        a = roc_auc_score(y, sum(wi * R[k] for wi, k in zip(ww, ks)))
        if a > best[0]: best = (a, dict(zip(ks, np.round(ww, 3))))
else:
    best = (roc_auc_score(y, O[ks[0]]), {ks[0]: 1.0})
print(f'\nBEST BLEND OOF AUC = {best[0]:.5f}')
print('weights:', best[1])

t = sum(best[1][k] * RT[k] for k in ks)
t = (t - t.min()) / (t.max() - t.min())
pd.DataFrame({'id': test.id, 'addicted_label': t}).to_csv(f'{WORK}/submission.csv', index=False)
SUBS['blend'] = (best[0], f'{WORK}/submission.csv')
print(f'wrote {WORK}/submission.csv')


### All submission files


### このセルがやっていること（What）**提出ファイルの健全性チェック（sanity check）**です。全ての提出ファイルについて:- OOF AUCの一覧を降順で表示- 行数がテストと一致するか- `id` がテストと**同じ順序で完全一致**するか- 予測値に欠損（NaN）がないか- 予測値の平均### なぜそうするのか（Why）**このセルは地味ですが、Kaggleで最も価値のある習慣の1つです。**提出ファイルの事故は、**モデルの良し悪しと無関係にスコアを0にします**。よくある事故:- マージの副作用で**行の順序が入れ替わる** → idが揃っていても値が全部ずれる- 未知のキーで `merge` した結果 **NaN が混入**する- fold平均の割り算を間違えて**行数が合わない**これらは学習コードを何時間も見直しても見つかりません。**提出直前に機械的に検証する**のが唯一の確実な対策です。特に `(d.id.values == test.id.values).all()` という**順序込みの完全一致チェック**が重要。「idの集合が同じ」だけでは不十分で、`sort` や `groupby` が暗黙に順序を変えていた、という事故を捕まえられます。`mean` を表示しているのも有効で、**基準率とかけ離れていたら何かがおかしい**という早期警告になります。> このパターンは自分の notebook にもそのままコピーして使う価値があります。> 「提出前チェックセル」を常設するだけで、無駄な1日を何度も救えます。

In [ ]:
print(f'{"model":22s} {"OOF AUC":>9s}   file')
for k, (auc, path) in sorted(SUBS.items(), key=lambda kv: -kv[1][0]):
    print(f'{k:22s} {auc:9.5f}   {os.path.basename(path)}')
print()
for k, (auc, path) in SUBS.items():
    d = pd.read_csv(path)
    ok = len(d) == len(test) and bool((d.id.values == test.id.values).all()) and d.addicted_label.notna().all()
    print(f'{os.path.basename(path):38s} rows={len(d)} valid={ok} mean={d.addicted_label.mean():.4f}')
print()
print('Submit submission.csv for the blend, or any submission_<model>.csv on its own.')


## Notes on what did *not* work

All measured on identical folds with a fixed seed and `deterministic=True` (noise floor
sd 0.00005), so these are dead ends rather than untested guesses:

| idea | effect |
|---|---|
| missing indicators + `n_missing` | +0.00001 |
| model-based imputation + reconstruction residuals | +0.00015 |
| generic ratio features on imputed values | **negative** |
| constraint-slack features for missing components | +0.00001 |
| last-decimal-digit features (real at z=6.8, but too small) | **negative** |
| multi-scale target encoding (0.1 / 0.5 grids) | −0.00002 |
| value-level encoding of *derived* quantities | none (half-corr ≤0.12 vs 0.33–0.72 for raw columns) |
| `age` as a categorical | −0.0006 |
| kNN local-memorisation features (residual corr −0.002) | none |
| record linkage / duplicate rows | none exist (565,846 rows → 1 duplicate pair) |

The missing-data angle is worth essentially nothing here despite 61% of rows having a missing
value — LightGBM's native NaN handling is already adequate, and for the network the learned
NaN embedding covers it.

### One result that depends on the fold count

Compositional (log-ratio) coordinates of the budget — `log(social/work)`, `log(gaming/work)`,
centered log-ratios, composition entropy — gave **+0.00036 at 3-fold** but **+0.00001 at
10-fold**. With 90% of the data instead of 67%, the trees learn those relationships from
splits by themselves and the explicit features stop paying. They are deliberately *not*
included here.

The budget constraint itself does **not** behave that way — ablating it at 10-fold costs
+0.00096 per fold (0.96782/0.96757/0.96824/0.96832/0.96803 with it, versus
0.96681/0.96647/0.96729/0.96753/0.96707 without), matching its 3-fold measurement of
+0.00092. A 4-term linear combination is something axis-aligned splits cannot build at any
data volume, which is why that one survives and the log-ratios do not.

The general lesson: validate feature-engineering gains at the fold count you will actually
submit with.
